In [1]:
import pandas as pd
import numpy as np
import joblib
from mcv1_forecast.core import train_model, recursive_forecast, COUNTRIES, mape
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings("ignore")

df_raw = pd.read_csv("vaccine_data.csv")

In [2]:
# Train and validate (backtest 2020-2024)
split_year = 2020
model, df_full, feature_cols, dummy_cols = train_model(df_raw, split_year=split_year)

TimeSeries CV MAE Scores: [53.64909364 25.9141091   6.2234409 ]
Average CV MAE: 28.596


In [3]:
# Validate
rec_results = recursive_forecast(df_full, model, feature_cols, dummy_cols, split_year)
#metrices
print("Recursive backtest (2020-2024)")
y_t = rec_results["Actual"].values
y_p = rec_results["Predicted"].values
print(f"MAE : {mean_absolute_error(y_t, y_p):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_t, y_p)):.3f}")
print(f"MAPE: {mape(y_t, y_p):.3f}%")

print(f"\n{'Country':<15} {'MAE':>8} {'RMSE':>8} {'MAPE%':>8}")
for country in COUNTRIES:
    sub = rec_results[rec_results["Country"] == country]
    y_t_c, y_p_c = sub["Actual"].values, sub["Predicted"].values
    print(f"{country:<15} {mean_absolute_error(y_t_c,y_p_c):>8.3f} {np.sqrt(mean_squared_error(y_t_c,y_p_c)):>8.3f} {mape(y_t_c,y_p_c):>8.3f}")


Recursive backtest (2020-2024)
MAE : 7.243
RMSE: 9.062
MAPE: 3.047%

Country              MAE     RMSE    MAPE%
Kyrgyzstan         5.699    7.092    2.888
Lesotho            2.932    4.236    5.132
Uzbekistan        13.097   13.346    1.120


In [5]:
# Retrain on all available data for future forecasting
final_split_year = 2025
final_model, final_df_full, final_feature_cols, final_dummy_cols = train_model(df_raw, split_year=final_split_year)

# Save the model and metadata for forecasting
joblib.dump({
    'model': final_model,
    'feature_cols': final_feature_cols,
    'dummy_cols': final_dummy_cols
}, 'mcv1_model.pkl')
print("Model saved to mcv1_model.pkl")

TimeSeries CV MAE Scores: [10.36511103  7.43322898 13.87630015]
Average CV MAE: 10.558
Model saved to mcv1_model.pkl
